# Stage 04x — Model Comparison

Compare three parallel model-building approaches (MIV, XGBoost Importance, Forward Stepwise),
compute weighted composite scores, and select the champion model.

In [ ]:
import sys, os
os.chdir('c:/projects/superagent')
sys.path.insert(0, 'src')
import pdtoolkit as pdt
import json, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
import statsmodels.api as sm
import shutil
import warnings
warnings.filterwarnings('ignore')

RUN_DIR = 'runs/2026-03-15_201852'
FIGURES = f'{RUN_DIR}/figures'
PIPELINE = f'{RUN_DIR}/pipeline'

## 1. Load Model Parameters

In [ ]:
with open(f'{PIPELINE}/model_params_miv.json') as f:
    params_miv = json.load(f)
with open(f'{PIPELINE}/model_params_xgb.json') as f:
    params_xgb = json.load(f)
with open(f'{PIPELINE}/model_params_fwd.json') as f:
    params_fwd = json.load(f)

models = {
    'MIV': params_miv,
    'XGBoost': params_xgb,
    'Forward': params_fwd
}

# Load binned data
df = pd.read_csv(f'{RUN_DIR}/data/loans_binned.csv')
y = df['Creditability'].astype(int)
print(f'Dataset: {len(df)} rows, target rate: {y.mean():.3f}')

## 2. Reconstruct Predictions for Each Model

In [ ]:
def woe_encode(df, woe_mappings, variables):
    """WoE-encode selected variables using model's mappings."""
    encoded = pd.DataFrame(index=df.index)
    for var in variables:
        mapping = {item['bin']: item['woe'] for item in woe_mappings[var]}
        encoded[var] = df[var].map(mapping)
        if encoded[var].isna().any():
            print(f'  Warning: {encoded[var].isna().sum()} unmapped values in {var}')
            encoded[var] = encoded[var].fillna(0.0)
    return encoded

def predict_proba(encoded, coefficients, intercept, variables):
    """Compute predicted probabilities from WoE-encoded data."""
    logit = intercept
    for var in variables:
        logit = logit + coefficients[var] * encoded[var]
    prob = 1.0 / (1.0 + np.exp(-logit))
    return prob

predictions = {}
for name, params in models.items():
    variables = params['selected_variables']
    encoded = woe_encode(df, params['woe_mappings'], variables)
    proba = predict_proba(encoded, params['coefficients'], params['intercept'], variables)
    predictions[name] = proba
    auc = roc_auc_score(y, proba)
    print(f'{name}: AUC = {auc:.4f} (stored: {params["model_auc"]:.4f}), vars = {len(variables)}')

## 3. Metrics Summary

In [ ]:
cv_aucs = {'MIV': 0.8201, 'XGBoost': 0.8134, 'Forward': 0.8201}

metrics = {}
for name, params in models.items():
    metrics[name] = {
        'AUC': params['model_auc'],
        'Gini': params['model_gini'],
        'KS': params['model_ks'],
        'CV AUC': cv_aucs[name],
        'N Variables': len(params['selected_variables']),
        'Method': params['selection_method']
    }

metrics_df = pd.DataFrame(metrics).T
print(metrics_df.to_string())

## 4. Weighted Composite Score

In [ ]:
def min_max_norm(values):
    """Min-max normalize; if all same, return 1.0 for all."""
    mn, mx = min(values), max(values)
    if mx - mn < 1e-10:
        return [1.0] * len(values)
    return [(v - mn) / (mx - mn) for v in values]

model_names = ['MIV', 'XGBoost', 'Forward']

aucs = [metrics[m]['AUC'] for m in model_names]
ginis = [metrics[m]['Gini'] for m in model_names]
kss = [metrics[m]['KS'] for m in model_names]
dev_aucs = [metrics[m]['AUC'] for m in model_names]
cv_auc_vals = [metrics[m]['CV AUC'] for m in model_names]
n_vars = [int(metrics[m]['N Variables']) for m in model_names]

# Sign consistency: all models have all-negative coefficients = consistent
sign_scores = [1.0, 1.0, 1.0]

# Decile monotonicity: all monotonic per stage summaries
mono_scores = [1.0, 1.0, 1.0]

# Normalized scores
auc_norm = min_max_norm(aucs)
gini_norm = min_max_norm(ginis)
ks_norm = min_max_norm(kss)

# CV stability: 1.0 - (abs(dev_auc - cv_auc) / 0.03), capped [0,1]
cv_stab = [max(0.0, min(1.0, 1.0 - abs(d - c) / 0.03)) for d, c in zip(dev_aucs, cv_auc_vals)]

# Variable parsimony: 1.0 - (n_vars - 4) / (12 - 4), capped [0,1]
parsimony = [max(0.0, min(1.0, 1.0 - (n - 4) / 8.0)) for n in n_vars]

# Composite
composites = []
for i in range(3):
    score = (0.25 * auc_norm[i] +
             0.15 * gini_norm[i] +
             0.10 * ks_norm[i] +
             0.20 * cv_stab[i] +
             0.15 * sign_scores[i] +
             0.10 * parsimony[i] +
             0.05 * mono_scores[i])
    composites.append(score)

score_detail = pd.DataFrame({
    'Model': model_names,
    'AUC (25%)': [f'{auc_norm[i]:.3f}' for i in range(3)],
    'Gini (15%)': [f'{gini_norm[i]:.3f}' for i in range(3)],
    'KS (10%)': [f'{ks_norm[i]:.3f}' for i in range(3)],
    'CV Stability (20%)': [f'{cv_stab[i]:.3f}' for i in range(3)],
    'Sign Consistency (15%)': [f'{sign_scores[i]:.3f}' for i in range(3)],
    'Parsimony (10%)': [f'{parsimony[i]:.3f}' for i in range(3)],
    'Monotonicity (5%)': [f'{mono_scores[i]:.3f}' for i in range(3)],
    'Composite': [f'{composites[i]:.4f}' for i in range(3)]
}).set_index('Model')

print(score_detail.to_string())
print()
for i, m in enumerate(model_names):
    print(f'{m}: composite = {composites[i]:.4f}')

## 5. Champion Selection

In [ ]:
champion_idx = np.argmax(composites)
champion_name = model_names[champion_idx]
champion_composite = composites[champion_idx]

# Determine runner-up (exclude champion index, pick highest remaining)
remaining = [(composites[i], model_names[i]) for i in range(3) if i != champion_idx]
remaining.sort(key=lambda x: -x[0])
runner_up_composite, runner_up_name = remaining[0]

suffix_map = {'MIV': 'miv', 'XGBoost': 'xgb', 'Forward': 'fwd'}
champion_suffix = suffix_map[champion_name]

print(f'Champion: {champion_name} (composite: {champion_composite:.4f})')
print(f'Runner-up: {runner_up_name} (composite: {runner_up_composite:.4f})')
print(f'Margin: {champion_composite - runner_up_composite:.4f}')

# Note: MIV and Forward have identical metrics and composite scores.
# MIV is selected as champion by convention (first in evaluation order).
if champion_composite == runner_up_composite:
    print(f'Note: {champion_name} and {runner_up_name} are tied. {champion_name} selected by convention.')

## 6. Visualizations

### 6.1 ROC Overlay

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
colors = {'MIV': '#2196F3', 'XGBoost': '#FF9800', 'Forward': '#4CAF50'}
linestyles = {'MIV': '-', 'XGBoost': '--', 'Forward': '-.'}

for name in model_names:
    fpr, tpr, _ = roc_curve(y, predictions[name])
    auc_val = metrics[name]['AUC']
    marker = ' *' if name == champion_name else ''
    ax.plot(fpr, tpr, color=colors[name], linestyle=linestyles[name],
            linewidth=2, label=f'{name} (AUC={auc_val:.4f}){marker}')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison - Three Selection Methods', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES}/04x_roc_overlay.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: 04x_roc_overlay.png')

### 6.2 Metrics Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

bar_metrics = ['AUC', 'Gini', 'KS']
x = np.arange(len(bar_metrics))
width = 0.25

for i, name in enumerate(model_names):
    vals = [metrics[name][m] for m in bar_metrics]
    bars = axes[0].bar(x + i * width, vals, width, label=name, color=colors[name], alpha=0.85)
    for bar, val in zip(bars, vals):
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                     f'{val:.4f}', ha='center', va='bottom', fontsize=8)

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(bar_metrics, fontsize=11)
axes[0].set_ylabel('Value', fontsize=11)
axes[0].set_title('Discrimination Metrics', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 1.0)
axes[0].grid(axis='y', alpha=0.3)

components = [
    ('AUC', 0.25, auc_norm),
    ('Gini', 0.15, gini_norm),
    ('KS', 0.10, ks_norm),
    ('CV Stab.', 0.20, cv_stab),
    ('Signs', 0.15, sign_scores),
    ('Parsimony', 0.10, parsimony),
    ('Monoton.', 0.05, mono_scores)
]

x2 = np.arange(len(model_names))
bottom = np.zeros(3)
comp_colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#795548', '#607D8B']

for j, (comp_name, weight, vals) in enumerate(components):
    weighted = [weight * v for v in vals]
    axes[1].bar(x2, weighted, 0.5, bottom=bottom, label=f'{comp_name} ({int(weight*100)}%)',
                color=comp_colors[j], alpha=0.85)
    bottom += weighted

for i, c in enumerate(composites):
    axes[1].text(i, c + 0.01, f'{c:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[1].set_xticks(x2)
axes[1].set_xticklabels(model_names, fontsize=11)
axes[1].set_ylabel('Weighted Score', fontsize=11)
axes[1].set_title('Composite Score Breakdown', fontsize=12)
axes[1].legend(fontsize=8, loc='upper right', ncol=2)
axes[1].set_ylim(0, 1.15)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES}/04x_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: 04x_model_comparison.png')

### 6.3 Variable Overlap Heatmap

In [ ]:
all_vars = sorted(set(
    params_miv['selected_variables'] +
    params_xgb['selected_variables'] +
    params_fwd['selected_variables']
))

overlap_matrix = pd.DataFrame(0, index=all_vars, columns=model_names)
for name, params in models.items():
    for var in params['selected_variables']:
        overlap_matrix.loc[var, name] = 1

overlap_matrix['Count'] = overlap_matrix.sum(axis=1).astype(int)
overlap_matrix = overlap_matrix.sort_values('Count', ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
plot_data = overlap_matrix[model_names].values.astype(float)
im = ax.imshow(plot_data, cmap='YlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(model_names, fontsize=11)
ax.set_yticks(range(len(overlap_matrix)))
ax.set_yticklabels(overlap_matrix.index, fontsize=10)

for i in range(plot_data.shape[0]):
    for j in range(plot_data.shape[1]):
        text = 'Yes' if plot_data[i, j] == 1 else 'No'
        color = 'white' if plot_data[i, j] == 1 else 'gray'
        ax.text(j, i, text, ha='center', va='center', fontsize=10, color=color, fontweight='bold')

ax.set_title('Variable Selection Overlap', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES}/04x_variable_overlap.png', dpi=150, bbox_inches='tight')
plt.close()

n_all = (overlap_matrix['Count'] == 3).sum()
n_any = len(overlap_matrix)
print(f'Variables selected by all 3 methods: {n_all}')
print(f'Total unique variables: {n_any}')
print(f'Overlap ratio: {n_all/n_any:.2f}')
print()
print(overlap_matrix.to_string())
print('\nSaved: 04x_variable_overlap.png')

## 7. Copy Champion Parameters

In [ ]:
src_path = f'{PIPELINE}/model_params_{champion_suffix}.json'
dst_path = f'{PIPELINE}/model_params.json'
shutil.copy2(src_path, dst_path)
print(f'Copied {src_path} -> {dst_path}')

with open(dst_path) as f:
    champion_params = json.load(f)
print(f'Champion: {champion_params["selection_method"]}')
print(f'Variables: {len(champion_params["selected_variables"])}')
print(f'AUC: {champion_params["model_auc"]:.4f}')

## 8. Summary

The MIV and Forward Stepwise methods selected identical variable sets (8 variables) and produced
identical model performance metrics. XGBoost Importance selected 7 variables (excluding Age),
resulting in slightly lower discrimination. The MIV method is selected as champion based on the
composite scoring framework, though it ties with Forward Stepwise on all metrics.